# Task patterns: classify / extract / summarize / generate

**Session 2 · small model (`llama3.2:3b`)**

Different tasks want different prompt shapes. Each shape also implies a different *check* —
a label set, a schema, a length, a forbidden phrase — so you can tell whether the output
actually did what you asked.

In [ ]:
import sys; sys.path.append('..')  # so `utils` and `eval` import from the repo root
import json
from utils import ask, SMALL_MODEL


### Worked example

One prompt for each of the four task types on the same input. Notice how the *shape* changes: label set vs schema vs length/audience vs style.


In [ ]:
# Worked example: classify / extract / summarise / generate on one input
SAMPLE = ("Order #4471 for Jane Cooper shipped on 2024-03-02 to Berlin. "
          "Total EUR 89.90. The customer says the box arrived dented.")

classify  = ask('Classify the message as: complaint, question, or praise. One word.\n\n'
                f'"{SAMPLE}"', model=SMALL_MODEL)
extract   = ask('Extract JSON with keys order_id, customer, city, total_eur. Return only JSON.\n\n'
                f'"{SAMPLE}"', model=SMALL_MODEL)
summarise = ask('Summarise in one sentence for a warehouse manager.\n\n'
                f'"{SAMPLE}"', model=SMALL_MODEL)
generate  = ask('Write a 2-sentence apology email. Warm. Do NOT promise a refund, replacement, '
                'or any specific compensation.\n\n'
                f'Context: "{SAMPLE}"', model=SMALL_MODEL)

for name, out in [("CLASSIFY", classify), ("EXTRACT", extract),
                  ("SUMMARISE", summarise), ("GENERATE", generate)]:
    print(f"--- {name} ---\n{out.strip()}\n")


### Each shape has its own check

The check is the point of the shape. `generate` is the interesting one: the model wants to be
helpful and offer a replacement — exactly what we forbade. Small models break this constraint
often; run the cell a few times.

In [ ]:
FORBIDDEN = ["refund", "replace", "replacement", "reship", "compensat", "voucher", "discount"]

print("CLASSIFY  in label set:  ", classify.strip().lower() in {"complaint", "question", "praise"})

try:
    obj = json.loads(extract[extract.find("{"): extract.rfind("}") + 1])
    print("EXTRACT   keys match:    ", set(obj) == {"order_id", "customer", "city", "total_eur"})
except (json.JSONDecodeError, ValueError):
    print("EXTRACT   keys match:     False (not valid JSON)")

print("SUMMARISE one sentence:  ", summarise.strip().rstrip(".").count(".") == 0)

hit = [w for w in FORBIDDEN if w in generate.lower()]
print("GENERATE  no promise:    ", not hit, f"(found: {hit})" if hit else "")


## Your turn - vary the example

1. Swap in your own SAMPLE (a review, a log line, an email) and re-run all four.
2. For EXTRACT, add a field and tighten the schema; for GENERATE, change the style constraint.
3. Which task type needs temperature 0? Which tolerates higher? Say why.
